# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/python/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# For metadata summary, use dataset.metadata attributes safely
print("Dataset loaded!")
print(f"Title: {dataset.metadata.name}\n")
print("Description:")
print(dataset.metadata.description)

## 2. Data Overview

Review available record sets, fields, and their IDs. All entities are referenced by their `@id` fields as per Croissant schema.

Let's list all record sets along with their IDs and fields. If your dataset only has one major record set, we'll use that for the examples below.

In [ ]:
record_sets = list(dataset.record_sets)
if len(record_sets) == 0:
    print("No record sets found in the metadata. Please check the schema for available record sets.")
else:
    print(f"Number of record sets found: {len(record_sets)}\n")
    for rs in record_sets:
        print(f"- RecordSet @id: {rs['@id']}  |  name: {rs.get('name', rs['@id'])}")
        if 'field' in rs:
            print("  Fields:")
            fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
            for fld in fields:
                fid = fld['@id'] if isinstance(fld, dict) and '@id' in fld else fld
                print(f"    - Field @id: {fid}")

Now, let's display a few example records from the **primary record set** (the central table containing the subjects/observations), referencing the record set by its `@id`.

If your dataset has only one main record set, use it in the code below.

In [ ]:
# Identify the main record set @id (copy from previous cell output):
# For this FAIR^2 dataset, let's find the ID programmatically then preview a few records.
main_record_set_id = None
if len(record_sets) > 0:
    # Use first available as example; replace below if you know exact one to use
    main_record_set_id = record_sets[0]['@id']
    print(f"Using RecordSet with @id: {main_record_set_id}\n")
    ex_rows = []
    for idx, rec in enumerate(dataset.records(record_set=main_record_set_id)):
        ex_rows.append(rec)
        if idx > 4:
            break
    if ex_rows:
        pd.set_option('display.max_columns', 50)
        display(pd.DataFrame(ex_rows))
    else:
        print("No records found for this RecordSet.")
else:
    print("No record sets could be previewed.")

## 3. Data Extraction

Load data from each record set into a pandas `DataFrame` for downstream analysis. Always use the record set and field `@id`s.

Below, we extract all tabular (primary) record sets as DataFrames.

In [ ]:
# Gather all record set @id's
all_record_set_ids = [rs['@id'] for rs in record_sets]
print(f"Available record sets: {all_record_set_ids}")

dataframes = {}
for rec_set_id in all_record_set_ids:
    records = list(dataset.records(record_set=rec_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rec_set_id] = df
        print(f"DataFrame created for RecordSet @id: {rec_set_id}  --> shape: {df.shape}")
    else:
        print(f"No records in RecordSet @id: {rec_set_id}")

# Show columns for main record set
if main_record_set_id and main_record_set_id in dataframes:
    print(f"\nColumns in main record set (@id: {main_record_set_id}):\n{dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())
else:
    print("Main record set dataframe not available.")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps: filtering numeric fields, normalization, and group-by operations. 

Suppose from the column list above, there is a numeric field (e.g., age or diagnosis_interval_years) you wish to analyze. Let's proceed with an example for a commonly available numeric field. Replace `diagnosis_interval_years` with the actual `@id` of your chosen field, as appropriate.

For demonstration, also show grouping by a categorical variable (e.g., sex, cancer_type, or similar—again, always using the @id as the key).

In [ ]:
# Example: Let's assume a field called diagnosis_interval_years (replace if needed)
# Set proper field @id based on actual data
numeric_field_id = None
cat_field_id = None

if main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    # Attempt to pick a numeric column
    for col in df.columns:
        # Attempt to find a numeric column by name hinting, can replace with actual @id
        if 'interval' in col or 'age' in col or 'years' in col:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break
    if not numeric_field_id:
        # Fallback: pick any numeric-type column
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break
    if not numeric_field_id:
        print("No numeric field found for EDA.")

    # Attempt to pick a group-by field
    for col in df.columns:
        if col != numeric_field_id and df[col].nunique() > 1 and (('sex' in col.lower()) or ('type' in col.lower()) or ('group' in col.lower())):
            cat_field_id = col
            break
    if not cat_field_id:
        # fallback, pick any non-numeric field with multiple categories
        for col in df.columns:
            if col != numeric_field_id and not pd.api.types.is_numeric_dtype(df[col]) and df[col].nunique() > 1:
                cat_field_id = col
                break
    
    if numeric_field_id:
        threshold = df[numeric_field_id].mean()  # Example: use mean as threshold
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize that numeric field
        filtered_df = filtered_df.copy()
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Optional: Group by a category if suitable
        if cat_field_id:
            print(f"\nGrouping by: {cat_field_id}")
            grouped_df = filtered_df.groupby(cat_field_id)[numeric_field_id].mean().reset_index().rename(columns={numeric_field_id: f"mean_{numeric_field_id}"})
            display(grouped_df.head())
        else:
            print("No suitable categorical field found for group-by analysis.")
    else:
        print("No numeric fields found in the main RecordSet. Please update field selection.")
else:
    print("Main RecordSet DataFrame is not available.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset. We'll plot a histogram of the selected numeric field (if available).

If grouping was possible, also create a bar plot of group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id in dataframes and numeric_field_id:
    df = dataframes[main_record_set_id]
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if cat_field_id:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=cat_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {cat_field_id}")
        plt.xlabel(cat_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No valid data available for visualization.")

## 6. Conclusion

In this notebook, we demonstrated how to load and explore a FAIR^2-compliant clinical dataset using the `mlcroissant` library. 

We accessed dataset metadata, listed record sets and fields by `@id`, loaded record sets as DataFrames, and performed exploratory analysis with filtering, normalization, and grouping. Visualizations illustrated data distributions and categorical relationships.

**Key Takeaways:**
- The Croissant schema and `mlcroissant` library allow robust and reproducible access to FAIR datasets.
- Always reference record sets, fields, and columns by their `@id` for clarity and automatic compatibility.
- Further analysis can now proceed (e.g., modeling, statistical inference, ML pipelines) using the curated DataFrames.

*For more, see the [Croissant specification](https://mlcommons.github.io/croissant/).*